In [1]:
import pandas as pd

df = pd.read_csv("../data/ecommerce_sales.csv")

df.head()

,order_id,date,city,category,price,units,revenue
0,100000,2024-04-19,Stockholm,Sports,530.95,4,2123.80
1,100001,2024-05-06,Malmö,Sports,653.28,3,1959.84
2,100002,2024-03-07,Uppsala,Clothing,177.98,4,711.92
3,100003,2024-04-08,Göteborg,Home,348.57,3,1045.71
4,100004,2024-01-18,Uppsala,Electronics,777.07,3,2331.21


In [2]:
# Check column names

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

In [3]:
# Look at the structure

print("Rows and columns:", df.shape)
df.info()  # show column names, data types and non-empty counts

Rows and columns: (2500, 7)
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  2500 non-null   int64  
 1   date      2500 non-null   str    
 2   city      2500 non-null   str    
 3   category  2500 non-null   str    
 4   price     2500 non-null   float64
 5   units     2500 non-null   int64  
 6   revenue   2500 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 136.8 KB


In [4]:
# Check data quality

print("Missing values per column:")
print(df.isna().sum()) # count empty cells in each column

print("Duplicate rows:", df.duplicated().sum()) # count rows that are exact copies
print("Duplicate order_id:", df["order_id"].duplicated().sum()) # each order_id should appear only once

revenue_difference = (df["price"] * df["units"] - df["revenue"]).abs() # gap between price*units and revenue (always positive)
print("Max difference between price*units and revenue:", revenue_difference.max())

Missing values per column:
order_id    0
date        0
city        0
category    0
price       0
units       0
revenue     0
dtype: int64
Duplicate rows: 0
Duplicate order_id: 0
Max difference between price*units and revenue: 1.8189894035458565e-12


In [5]:
# Check for missing dates in the range

print("Before:", df["date"].dtype) # show the type before converting

df["date"] = pd.to_datetime(df["date"]) # turn text into real dates
#df = df.sort_values("date").reset_index(drop=True) # sort oldest to newest and restart row numbers at 0

print("After:", df["date"].dtype)
print("First date:", df["date"].min())
print("Last date:", df["date"].max())

expected_days = (df["date"].max() - df["date"].min()).days + 1  # days from first to last date
print("Days with sales:", df["date"].nunique(), "of", expected_days)

Before: str
After: datetime64[us]
First date: 2024-01-01 00:00:00
Last date: 2024-06-30 00:00:00
Days with sales: 182 of 182


In [6]:
# Create month columns for sorting and reading

df["month_number"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df[["date", "month_number", "month_name"]].head()

,date,month_number,month_name
0,2024-04-19,4,April
1,2024-05-06,5,May
2,2024-03-07,3,March
3,2024-04-08,4,April
4,2024-01-18,1,January


In [7]:
# Sales by month

monthly_summary = (
    df.groupby(["month_number", "month_name"])  # one group per month
    .agg(
        total_revenue = ("revenue", "sum"),
        total_units = ("units", "sum"),
        order_count = ("order_id", "count")
    )
    .reset_index()  # turn the group labels back into normal columns
)

monthly_summary["average_order_value"] = monthly_summary["total_revenue"] / monthly_summary["order_count"]
monthly_summary = monthly_summary.round(2)
monthly_summary = monthly_summary.drop(columns = "month_number")  # show without month number

print(monthly_summary)

  month_name  total_revenue  total_units  order_count  average_order_value
0    January      664083.32         1353          445              1492.32
1   February      548462.03         1097          357              1536.31
2      March      487602.00         1116          411              1186.38
3      April      580194.83         1305          432              1343.04
4        May      632165.16         1367          433              1459.97
5       June      590259.89         1225          422              1398.72


In [8]:
# Create weekday columns for sorting and reading

df["weekday_number"] = df["date"].dt.weekday +1
df["day_name"] = df["date"].dt.day_name()
df[["date", "weekday_number", "day_name"]].head()


,date,weekday_number,day_name
0,2024-04-19,5,Friday
1,2024-05-06,1,Monday
2,2024-03-07,4,Thursday
3,2024-04-08,1,Monday
4,2024-01-18,4,Thursday


In [9]:
# Sales by weekday

weekday_summary = (
    df.groupby(["weekday_number", "day_name"])  # one group per weekday
    .agg(
        total_revenue = ("revenue", "sum"),
        total_units = ("units", "sum"),
        order_count = ("order_id", "count")
    )
    .reset_index()  # turn the group labels back into normal columns
)

weekday_summary["average_order_value"] = weekday_summary["total_revenue"] / weekday_summary["order_count"]
weekday_summary = weekday_summary.round(2)
weekday_summary = weekday_summary.drop(columns = "weekday_number")  # show without weekday number

print(weekday_summary)

    day_name  total_revenue  total_units  order_count  average_order_value
0     Monday      527263.73         1115          368              1432.78
1    Tuesday      512978.35         1037          354              1449.09
2  Wednesday      502661.61         1063          346              1452.78
3   Thursday      518871.71         1098          361              1437.32
4     Friday      501396.53         1082          357              1404.47
5   Saturday      481483.77         1052          355              1356.29
6     Sunday      458111.53         1016          359              1276.08


In [10]:
# Sales by week

weekly_summary = (
    df.resample("W", on="date")  # cut the timeline into weeks (Monday to Sunday) using the date column
    .agg(
        total_revenue = ("revenue", "sum"),
        total_units = ("units", "sum"),
        order_count = ("order_id", "count")
    )
    .reset_index()  # turn the week labels back into normal columns
    .rename(columns = {"date": "week_ending"})  # rename it: the label is the Sunday that ends each week
)

weekly_summary["average_order_value"] = weekly_summary["total_revenue"] / weekly_summary["order_count"]
weekly_summary = weekly_summary.round({"total_revenue": 2,"average_order_value": 2})  # round only these columns

print("Number of weeks:", len(weekly_summary))
weekly_summary.head()

Number of weeks: 26


,week_ending,total_revenue,total_units,order_count,average_order_value
0,2024-01-07,161119.04,326,108,1491.84
1,2024-01-14,136568.68,265,81,1686.03
2,2024-01-21,147225.11,319,104,1415.63
3,2024-01-28,165832.80,328,108,1535.49
4,2024-02-04,124154.34,247,89,1394.99
